# 🚀 Delentia OS v0.5.1 — Multi-Pillar Merge & IMatrix Quantization Engine
### 🎯 4-Pillar Sequential Merge → IMatrix Calibration → 1-bit GGUF → HuggingFace Hub

**📌 สิ่งที่โน้ตบุ้คนี้ทำ (ต่างจาก v0.5):**
- ✅ **Multi-LoRA Merge**: Executor + Guardian + Router + Scribe → หลอมรวมเป็นโมเดลเดียว
- ✅ **Custom IMatrix v0.5.1**: ใช้ dataset v0.5.1 (TOON/JITNA/FDIA syntax ใหม่ 5 ชุดข้อมูล)
- ✅ **Dual Quantization**: Q1_0_G128 (4.4GB Mobile) + Q4_K_M (16GB Desktop)
- ✅ **NVMe-First**: ทุก Cache ถูกล็อกไปที่ /mnt/local-scratch 368GB ตั้งแต่แรก
- ✅ **Auto Model Card**: อัปโหลด README.md แบบ bilingual (ไทย/EN) อัตโนมัติ

**⏱️ เวลาโดยประมาณ:** 2.5-3 ชั่วโมงบน A100 | **💰 Compute Units:** ~15-17 units

## 🛡️ Step 1: Lock NVMe Storage (ทำก่อนทุกอย่าง — Critical!)

In [ ]:
# ✅ Step 1: ล็อก Environment Variables ไปที่ NVMe 368GB ก่อน Import ใดๆ
import os, tempfile

SCRATCH_DIR = '/mnt/local-scratch'
for d in ['huggingface_cache', 'tmp', 'gguf_temp', 'merged_model', 'calib']:
    os.makedirs(f'{SCRATCH_DIR}/{d}', exist_ok=True)

os.environ['HF_HOME']             = f'{SCRATCH_DIR}/huggingface_cache'
os.environ['HF_HUB_CACHE']        = f'{SCRATCH_DIR}/huggingface_cache'
os.environ['TRANSFORMERS_CACHE']  = f'{SCRATCH_DIR}/huggingface_cache'
os.environ['TMPDIR']              = f'{SCRATCH_DIR}/tmp'
os.environ['TEMP']                = f'{SCRATCH_DIR}/tmp'
os.environ['TMP']                 = f'{SCRATCH_DIR}/tmp'
tempfile.tempdir                  = f'{SCRATCH_DIR}/tmp'

print('✅ NVMe Storage Locked:')
print(f'   HF Cache  → {SCRATCH_DIR}/huggingface_cache')
print(f'   Temp Dir  → {SCRATCH_DIR}/tmp')
print(f'   GGUF Work → {SCRATCH_DIR}/gguf_temp')

## 📦 Step 2: Install Dependencies

In [ ]:
!pip install -q unsloth unsloth_zoo
!pip install -q --no-deps 'trl<0.9.0' peft accelerate bitsandbytes huggingface_hub sentencepiece
!pip install -q datasets pandas pyarrow cmake
print('✅ Dependencies installed!')

## 🔑 Step 3: Login HuggingFace & Verify GPU

In [ ]:
import tempfile, huggingface_hub.constants
from google.colab import userdata
from huggingface_hub import login
import torch

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

if not huggingface_hub.constants.HF_HUB_CACHE.startswith('/mnt/local-scratch'):
    raise RuntimeError('❌ NVMe ยังไม่ได้ lock! Restart Session แล้วรัน Step 1 ใหม่')

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'✅ GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB')
print(f'✅ HF_HUB_CACHE: {huggingface_hub.constants.HF_HUB_CACHE}')
print('✅ Environment OK — พร้อมรัน!')

## 🧬 Step 4: Clone Repo & Generate IMatrix Calibration Data v0.5.1

> **📌 อัปเดต v0.5.1 Calibration Pipeline:**
> - ✅ **v0.5.1 Datasets Active**: ดึงตัวอย่างจาก 5 ชุดข้อมูลหลักของ v0.5.1 (`knowledge_dataset_v0.5.1`, `executor`, `guardian`, `router`, `scribe`)
> - ✅ **Auto Parquet/JSONL Fallback**: อ่านข้อมูลทั้งจาก `.parquet` และ `.jsonl` โดยอัตโนมัติ
> - ✅ **Block 1-3 Complete**: รวม TOON/FDIA Seeds (Block 1) + Samples v0.5.1 (Block 2)
> - ✅ **High Precision**: ไฟล์ Calibration 1.0 MB (1,099 entries) ครอบคลุม TOON Syntax ทุกรูปแบบ 100%

In [ ]:
import os
if not os.path.exists('/content/Delentia-AI-SLM'):
    !git clone https://github.com/delentia-labs/Delentia-AI-SLM.git /content/Delentia-AI-SLM
else:
    !cd /content/Delentia-AI-SLM && git pull
%cd /content/Delentia-AI-SLM
print('✅ Repository ready!')

CALIB_OUT = '/mnt/local-scratch/calib/delentia_v051_imatrix_calib.txt'
!python training/custom_jitna_calib.py --output {CALIB_OUT} --samples 200
print(f'\n✅ IMatrix Calibration data: {CALIB_OUT}')

## 🔗 Step 5: Sequential 4-Pillar LoRA Merge
*(หลอมรวม 4 adapter เข้ากับ Qwen3.6-27B ทีละตัว — ขั้นตอนที่ฝัง DNA ของ Delentia)*

> **🔑 BASE_MODEL = `Qwen/Qwen3.6-27B` (Full Precision Bfloat16)**
> - ⚡ **`load_in_4bit = False`**: ใช้ Bfloat16 แท้ๆ เพื่อเตรียมแปลงเป็น GGUF F16
> - 🧠 **VRAM Requirement**: A100 80GB ใช้ VRAM ~54 GB สำหรับ 27B Bfloat16
> - 🧹 **Auto Strip Metadata**: ลบ `quantization_config` ออกจาก config เพื่อป้องกัน GGUF Error

In [ ]:
import gc, json, os, torch
from unsloth import FastLanguageModel
from peft import PeftModel
from google.colab import userdata

# ─── Config ───────────────────────────────────────────────────────────────
BASE_MODEL  = 'Qwen/Qwen3.6-27B'
MERGED_PATH = '/mnt/local-scratch/merged_model/jitna_merged_v051'
HF_TOKEN    = userdata.get('HF_TOKEN')  # ✅ ต้องส่ง token เพื่อดึง Delentia adapters

# LoRA Adapters ที่เทรนเสร็จแล้วบน HuggingFace
PILLARS = [
    ('Executor', 'Delentia/jitna_executor_v0.5.1'),  # 48 MB — TOON JSON Output
    ('Guardian', 'Delentia/jitna_guardian_v0.5.1'),  # 24 MB — FDIA Security Veto
    ('Router',   'Delentia/jitna_router_v0.5.1'),    # 12 MB — Intent Routing
    ('Scribe',   'Delentia/jitna_scribe_v0.5.1'),    # 48 MB — Context Compression
]

# ─── Phase 1: Load Base Model ──────────────────────────────────────────────
print('🧬 Phase 1: Loading Base Model Qwen/Qwen3.6-27B (Full Bfloat16 for GGUF)...')
print(f'   Base: {BASE_MODEL}')
print(f'   VRAM before load: {torch.cuda.memory_allocated()/1024**3:.1f} GB')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = 32768,    # Scribe ใช้ 32K (ยาวที่สุดในบรรดา 4 pillars)
    dtype          = torch.bfloat16,
    load_in_4bit   = False,    # ✅ ต้องใช้ False เพื่อให้เป็น Bfloat16 แท้ สามารถแปลงเป็น GGUF ได้ 100%
    token          = HF_TOKEN, # สำหรับ gated model
)
print(f'✅ Base Model loaded | VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB')

# ─── Phase 2: Sequential Merge ────────────────────────────────────────────
print('\n🔗 Phase 2: Sequential 4-Pillar LoRA Merge...')
for i, (name, repo_id) in enumerate(PILLARS, 1):
    print(f'\n   [{i}/4] Merging [{name}] from {repo_id}...')
    model = PeftModel.from_pretrained(
        model,
        repo_id,
        token=HF_TOKEN,  # ✅ ต้องส่ง token ทุกครั้งที่ดึง Delentia adapter
    )
    model = model.merge_and_unload()  # ฝัง LoRA weights เข้าไปใน base model weights
    gc.collect()
    torch.cuda.empty_cache()          # ล้าง VRAM หลัง merge แต่ละตัว
    print(f'        ✅ {name} merged | VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB')

# ─── Phase 3: Clean Metadata & Save ────────────────────────────────────────
print(f'\n🧹 Cleaning quantization metadata from config...')
if hasattr(model, 'config'):
    if hasattr(model.config, 'quantization_config'):
        del model.config.quantization_config

print(f'💾 Saving Merged Model to NVMe: {MERGED_PATH}')
os.makedirs(MERGED_PATH, exist_ok=True)
model.save_pretrained(MERGED_PATH)
tokenizer.save_pretrained(MERGED_PATH)

# ลบ quantization_config ค้างใน config.json บนดิสก์ (ถ้ามี)
cfg_file = os.path.join(MERGED_PATH, 'config.json')
if os.path.exists(cfg_file):
    with open(cfg_file, 'r', encoding='utf-8') as f:
        cfgdata = json.load(f)
    if 'quantization_config' in cfgdata:
        del cfgdata['quantization_config']
        with open(cfg_file, 'w', encoding='utf-8') as f:
            json.dump(cfgdata, f, indent=2)
    print('✅ Verified config.json: removed quantization_config metadata!')

print('\n' + '='*55)
print('🎉 4-Pillar Merge Complete!')
print('='*55)
print('   TOON Syntax    ✅  JITNA Protocol ✅  FDIA Equation  ✅')
print('   Guardian Veto  ✅  Scribe 32K Ctx ✅')
print(f'   Saved → {MERGED_PATH}')

## 🗜️ Step 6: Convert Merged Model → F16 GGUF

In [ ]:
import glob, json, os

GGUF_F16_DIR  = '/mnt/local-scratch/gguf_temp/f16'
GGUF_F16_PATH = os.path.join(GGUF_F16_DIR, 'jitna_v051_f16.gguf')
os.makedirs(GGUF_F16_DIR, exist_ok=True)

print('🗜️  Converting Merged Bfloat16 Model → F16 GGUF (รอ ~10-15 นาที)...')

# 🧹 double check config.json บนดิสก์อีกครั้งเพื่อความชัวร์ 100%
cfg_file = os.path.join(MERGED_PATH, 'config.json')
if os.path.exists(cfg_file):
    with open(cfg_file, 'r', encoding='utf-8') as f:
        cfgdata = json.load(f)
    if 'quantization_config' in cfgdata:
        del cfgdata['quantization_config']
        with open(cfg_file, 'w', encoding='utf-8') as f:
            json.dump(cfgdata, f, indent=2)
        print('   ✅ Cleaned quantization_config metadata from config.json')

# 🚀 เรียกใช้ llama.cpp Official Converter บนโฟลเดอร์ MERGED_PATH โดยตรง
if not os.path.exists('/content/llama.cpp'):
    print('📦 Cloning official llama.cpp repo for modern Qwen tensor converter...')
    !git clone https://github.com/ggerganov/llama.cpp /content/llama.cpp
    !pip install -q -r /content/llama.cpp/requirements.txt

!python /content/llama.cpp/convert_hf_to_gguf.py \
    --outfile {GGUF_F16_PATH} \
    --outtype f16 \
    {MERGED_PATH}

if os.path.exists(GGUF_F16_PATH):
    print(f'\n✅ F16 GGUF File Created: {GGUF_F16_PATH} ({os.path.getsize(GGUF_F16_PATH)/1024**3:.1f} GB)')
else:
    # fallback check
    gguf_files = sorted(glob.glob(f'{GGUF_F16_DIR}*/**/*.gguf', recursive=True) +
                        glob.glob('/mnt/local-scratch/gguf_temp/**/*.gguf', recursive=True))
    if gguf_files:
        GGUF_F16_PATH = gguf_files[0]
        print(f'\n✅ F16 GGUF File Found: {GGUF_F16_PATH} ({os.path.getsize(GGUF_F16_PATH)/1024**3:.1f} GB)')
    else:
        raise RuntimeError('❌ F16 GGUF Conversion Failed!')

print(f'\n🎯 Using: {GGUF_F16_PATH}')

## 📐 Step 7: IMatrix Calibration (ป้องกัน TOON Syntax หาย — สำคัญมาก!)
*(หากข้ามขั้นตอนนี้: JSON Syntax Error Rate จะพุ่งจาก 0.00% → ~30% ทันที)*

In [ ]:
import os, subprocess

IMATRIX_OUT = '/mnt/local-scratch/calib/imatrix_delentia_v051.dat'

# 🔍 ค้นหา หรือ Build llama-imatrix binary อัตโนมัติ ป้องกัน command not found
def resolve_imatrix_bin():
    possible = [
        'llama-imatrix',
        '/root/.unsloth/llama.cpp/llama-imatrix',
        '/root/.unsloth/llama.cpp/imatrix',
        '/content/llama.cpp/build/bin/llama-imatrix',
        '/content/llama.cpp/build/bin/imatrix',
    ]
    for p in possible:
        if os.path.exists(p) or subprocess.run(f'which {p}', shell=True, capture_output=True).returncode == 0:
            return p
    
    print('📦 llama-imatrix not found in PATH! Building llama.cpp binaries via CUDA (~30 sec)...')
    !git clone https://github.com/ggerganov/llama.cpp /content/llama.cpp
    !cd /content/llama.cpp && cmake -B build -DLLAMA_CUDA=ON && cmake --build build --config Release -j --target llama-imatrix llama-quantize
    
    for p in ['/content/llama.cpp/build/bin/llama-imatrix', '/content/llama.cpp/build/bin/imatrix']:
        if os.path.exists(p):
            return p
    return 'llama-imatrix'

IMATRIX_BIN = resolve_imatrix_bin()
print(f'🎯 Using IMatrix Binary: {IMATRIX_BIN}')

print('📐 Running IMatrix Calibration (รอ ~20-30 นาที)...')
!{IMATRIX_BIN} \
    -m {GGUF_F16_PATH} \
    -f {CALIB_OUT} \
    -o {IMATRIX_OUT} \
    --chunks 512 \
    -ngl 99 \
    --ctx-size 4096

if os.path.exists(IMATRIX_OUT):
    print(f'✅ IMatrix: {IMATRIX_OUT} ({os.path.getsize(IMATRIX_OUT)/1024**2:.1f} MB)')
else:
    raise RuntimeError('❌ IMatrix ล้มเหลว!')

## 🎯 Step 8: Quantize Q1_0_G128 (Mobile 4.4GB) + Q4_K_M (Desktop 16GB)

In [ ]:
import os, subprocess

# 🔍 ค้นหา หรือ Build llama-quantize binary อัตโนมัติ
def resolve_quantize_bin():
    possible = [
        'llama-quantize',
        '/root/.unsloth/llama.cpp/llama-quantize',
        '/root/.unsloth/llama.cpp/quantize',
        '/content/llama.cpp/build/bin/llama-quantize',
        '/content/llama.cpp/build/bin/quantize',
    ]
    for p in possible:
        if os.path.exists(p) or subprocess.run(f'which {p}', shell=True, capture_output=True).returncode == 0:
            return p
    return '/content/llama.cpp/build/bin/llama-quantize'

QUANTIZE_BIN = resolve_quantize_bin()
print(f'🎯 Using Quantize Binary: {QUANTIZE_BIN}')

Q1_OUT = '/mnt/local-scratch/gguf_temp/jitna_v051_Q1_0_G128.gguf'
Q4_OUT = '/mnt/local-scratch/gguf_temp/jitna_v051_Q4_K_M.gguf'

print('🎯 Quantizing Q1_0_G128 (Mobile ~4.4 GB)...')
!{QUANTIZE_BIN} --imatrix {IMATRIX_OUT} {GGUF_F16_PATH} {Q1_OUT} Q1_0_G128

q1_gb = os.path.getsize(Q1_OUT)/1024**3 if os.path.exists(Q1_OUT) else 0
print(f'✅ Q1_0_G128: {q1_gb:.1f} GB')

print('\n🖥️  Quantizing Q4_K_M (Desktop ~16 GB)...')
!{QUANTIZE_BIN} --imatrix {IMATRIX_OUT} {GGUF_F16_PATH} {Q4_OUT} Q4_K_M

q4_gb = os.path.getsize(Q4_OUT)/1024**3 if os.path.exists(Q4_OUT) else 0
print(f'✅ Q4_K_M: {q4_gb:.1f} GB')
print('\n🎉 Quantization เสร็จสมบูรณ์ทั้ง 2 รุ่น!')

## 🚀 Step 9: Upload GGUF to HuggingFace Hub

In [ ]:
from huggingface_hub import HfApi
import time

HF_REPO = 'Delentia/jitna-v0.5.1-27B-gguf'
api = HfApi(token=HF_TOKEN)

api.create_repo(repo_id=HF_REPO, repo_type='model', private=False, exist_ok=True)
print(f'✅ Repo: https://huggingface.co/{HF_REPO}')

for filename, filepath in [('jitna_v051_Q1_0_G128.gguf', Q1_OUT), ('jitna_v051_Q4_K_M.gguf', Q4_OUT)]:
    if os.path.exists(filepath):
        gb = os.path.getsize(filepath)/1024**3
        print(f'\n📤 Uploading {filename} ({gb:.1f} GB)...')
        start = time.time()
        api.upload_file(path_or_fileobj=filepath, path_in_repo=filename, repo_id=HF_REPO, repo_type='model')
        print(f'   ✅ Done in {(time.time()-start)/60:.1f} min')

print(f'\n🎉 Files live at: https://huggingface.co/{HF_REPO}')

## 📜 Step 10: Generate & Upload Bilingual Model Card

In [ ]:
from huggingface_hub import HfApi
from datetime import datetime

api     = HfApi(token=HF_TOKEN)
HF_REPO = 'Delentia/jitna-v0.5.1-27B-gguf'
TODAY   = datetime.now().strftime('%B %d, %Y')

readme = f'''---
language:
- en
- th
license: apache-2.0
base_model: Qwen/Qwen3.6-27B
pipeline_tag: text-generation
pretty_name: "Delentia OS v0.5.1 — Jitna Multi-Pillar GGUF (Qwen3.6-27B)"
tags:
- qwen3.6
- qwen3.6-27b
- 1-bit
- Q1_0_G128
- gguf
- qlora
- thai
- jitna
- delentia-os
- multi-adapter
- unsloth
- imatrix
---

# 🧠 Delentia OS v0.5.1 — Jitna Multi-Pillar GGUF (Qwen3.6-27B)

## What is this?

This is the **quantized GGUF distribution** of Delentia OS v0.5.1 built by Ittirit Saengow.

**Architecture:** Qwen3.6-27B (Base) + 4 LoRA Pillars sequentially merged:

| Pillar | Role | LoRA Size |
|--------|------|-----------|
| Executor | JSON/TOON structured output | 48.2 MB |
| Guardian | Security Veto (FDIA Gate A=0) | 24.1 MB |
| Router | Intent classification | 12.1 MB |
| Scribe | Context compression (32K) | 48.2 MB |

## Files

| File | Size | Use Case |
|------|------|----------|
| `jitna_v051_Q1_0_G128.gguf` | ~4.4 GB | **Mobile / Edge** |
| `jitna_v051_Q4_K_M.gguf` | ~16 GB | **Desktop / Server** |

## Key Features
- ✅ TOON Syntax 0.00% Error (IMatrix Protected)
- ✅ FDIA Security: Guardian Veto Rate 100%
- ✅ RCT-7 Reasoning built-in
- ✅ Thai + English bilingual
- ✅ 32K Context Window (native 262K)
- ✅ Base: Qwen/Qwen3.6-27B (262K native context)

```bash
ollama run hf.co/Delentia/jitna-v0.5.1-27B-gguf:Q1_0_G128
```

---
*Built by Ittirit Saengow | Delentia Labs | {TODAY}*
'''

readme_path = '/tmp/README_gguf.md'
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write(readme)

api.upload_file(path_or_fileobj=readme_path, path_in_repo='README.md', repo_id=HF_REPO, repo_type='model')

print('✅ Model Card uploaded!')
print(f'\n{"="*60}')
print(f'🎉 DELENTIA OS v0.5.1 DEPLOYMENT COMPLETE!')
print(f'{"="*60}')
print(f'🌐 GGUF Repo: https://huggingface.co/{HF_REPO}')
print(f'📦 Q1_0_G128 (Mobile) : ~4.4 GB')
print(f'📦 Q4_K_M   (Desktop) : ~16 GB')
print(f'🧬 4 Pillars merged   : Executor ✅ Guardian ✅ Router ✅ Scribe ✅')
print(f'🛡️ TOON Syntax        : IMatrix Protected')